In [ ]:
model_path = "./model"

ExecutionProvider="OpenVINOExecutionProvider"

In [ ]:
from winml import register_execution_providers_to_onnxruntime_genai
register_execution_providers_to_onnxruntime_genai()

In [ ]:
import requests

test_audio_url = "https://storage.openvinotoolkit.org/models_contrib/speech/2021.2/librispeech_s5/how_are_you_doing_today.wav"
test_audio_name = "how_are_you_doing_today.wav"

r = requests.get(test_audio_url)
with open(test_audio_name, "wb") as audio_file:
    audio_file.write(r.content)

from IPython.display import Audio, display

display(Audio(filename=test_audio_name))

In [ ]:
from pathlib import Path
import onnxruntime_genai as og

def generate_transcript(model_path, audio_path, num_beams=0):
    """Generate transcript using onnxruntime-genai.
    Args:
        model_path: Path to the genai model directory
        audio_path: Path to audio file
        num_beams: Number of beams for beam search
        execution_provider: Execution provider (cpu, cuda, or follow_config)
    Returns:
        Transcription text
    """
    print("Loading model...")
    print(f"Model path: {model_path}")
    config = og.Config(model_path)
    model = og.Model(config)
    processor = model.create_multimodal_processor()

    print("Loading audio...")
    if not Path(audio_path).exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")
    audios = og.Audios.open(audio_path)
    
    print(f"Processing audio: {audio_path}")
    batch_size = 1
    decoder_prompt_tokens = ["<|startoftranscript|>", "<|en|>", "<|transcribe|>", "<|notimestamps|>"]
    prompts = ["".join(decoder_prompt_tokens)]
    inputs = processor(prompts, audios=audios)

    print(f"Processing:")
    params = og.GeneratorParams(model)
    params.set_search_options(
        do_sample=False,
        num_beams=num_beams,
        num_return_sequences=num_beams,
        max_length=448,
        batch_size=batch_size,
    )

    generator = og.Generator(model, params)
    generator.set_inputs(inputs)

    while not generator.is_done():
        generator.generate_next_token()

    print("Finish generation. Decoding outputs...")
    transcriptions = []
    for i in range(batch_size * num_beams):
        tokens = generator.get_sequence(i)
        transcription = processor.decode(tokens)
        transcriptions.append(transcription.strip())

    return transcriptions[0]

num_beams = 1
transcript = generate_transcript(model_path, test_audio_name, num_beams)
print("Transcript: ", transcript)